# 🤖 AI Engineering Fundamentals — Lezione 1
## Notebook Gruppo A

**ITS Novitas 4.0 | Martedì 19/05/2026**

---

### 📋 Istruzioni
1. Cliccate **File → Salva una copia in Drive** prima di iniziare
2. Configurate la API key nei Secrets (🔑 nella barra sinistra)
3. Lavorate in gruppo — discutete le risposte prima di scrivere
4. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo
Scrivete i vostri nomi qui sotto:

In [ ]:
GRUPPO = "A"
MEMBRI = [
    "",  # ← Nome 1
    "",  # ← Nome 2
    "",  # ← Nome 3
    "",  # ← Nome 4 (se presente)
]

print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente

def chiedi_claude(messaggio, temperature=0.7, system=None, max_tokens=500):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo A: Token e Costi

Il vostro gruppo esplora come il testo viene trasformato in token
e come questo impatta i costi delle API.

---
### Esercizio 1 — Esplorare i token *(guidato)*

Ogni chiamata all'API restituisce anche il numero di token usati.
Completate il codice qui sotto per stampare input e output token.

In [ ]:
# Esercizio 1 — completate le parti mancanti

risposta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[{"role": "user", "content": "Cos'è l'intelligenza artificiale? Rispondi in 2 frasi."}]
)

# Il testo della risposta
print("Risposta:", risposta.content[0].text)

# Il numero di token in input
print("Token input:", risposta.usage.input_tokens)

# Il numero di token in output
print("Token output:", risposta.usage.output_tokens)

# Costo stimato
# Haiku: $1 per milione di token input, $5 per milione di token output
costo = (risposta.usage.input_tokens / 1_000_000 * 1.0) + (risposta.usage.output_tokens / 1_000_000 * 5.0)
print(f"Costo stimato: ${costo:.6f}")

---
### Esercizio 2 — Italiano vs Inglese *(guidato)*

La stessa frase in italiano consuma più token che in inglese.
Verificatelo empiricamente.

In [ ]:
# Esercizio 2 — confronto token italiano vs inglese

frase_it = "L'intelligenza artificiale sta cambiando il mondo del lavoro in modo profondo."
frase_en = "Artificial intelligence is deeply changing the world of work."

def conta_token(testo):
    """Conta i token di un testo senza inviarlo al modello."""
    result = client.messages.count_tokens(
        model="claude-haiku-4-5-20251001",
        messages=[{"role": "user", "content": testo}]
    )
    return result.input_tokens

token_it = conta_token(frase_it)
token_en = conta_token(frase_en)

print(f"Frase IT: '{frase_it}'")
print(f"Token IT: {token_it}")
print()
print(f"Frase EN: '{frase_en}'")
print(f"Token EN: {token_en}")
print()
print(f"Differenza: {token_it - token_en} token in più per l'italiano")
print(f"Overhead: +{((token_it/token_en)-1)*100:.0f}%")

# Discussione: il risultato vi sorprende? Perché secondo voi succede?
# Scrivete la vostra risposta come commento qui sotto:
# ...

---
### Esercizio 3 — Quanto costa una conversazione? *(libero)*

Simulate una conversazione di **5 domande e risposte** sul tema
dei prodotti WiData. Alla fine calcolate il costo totale.

**Domande da fare:**
1. Cosa fa WiData?
2. Che tipo di sensori offrite?
3. Funzionano all'aperto?
4. Come si connettono al cloud?
5. Quanto costano?

*(Non preoccupatevi se le risposte non sono accurate — Claude non conosce WiData. Lo correggeremo con RAG nella Lezione 4.)*

In [ ]:
# Esercizio 3 — simulate la conversazione e calcolate il costo totale

domande = [
    "Cosa fa WiData?",
    "Che tipo di sensori offrite?",
    "Funzionano all'aperto?",
    "Come si connettono al cloud?",
    "Quanto costano?",
]

token_input_totale = 0
token_output_totale = 0

# Per ogni domanda: chiamiamo l'API, stampiamo domanda e risposta, accumuliamo i token
for domanda in domande:
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": domanda}],
    )
    testo = risposta.content[0].text
    print(f"\n❓ {domanda}")
    print(f"🤖 {testo}")

    token_input_totale += risposta.usage.input_tokens
    token_output_totale += risposta.usage.output_tokens

# Costo totale (Haiku: $1/M input, $5/M output)
costo_totale = (token_input_totale / 1_000_000 * 1.0) + (token_output_totale / 1_000_000 * 5.0)

print("\n" + "=" * 50)
print(f"Token input totali:  {token_input_totale}")
print(f"Token output totali: {token_output_totale}")
print(f"Costo totale stimato: ${costo_totale:.6f}")

# Osservazione: ogni domanda è indipendente (non c'è memoria della conversazione).
# Nella Lezione 3 vedremo che inviare tutta la cronologia ad ogni turno fa
# crescere i token di input in modo cumulativo — e quindi anche il costo.

---
### Esercizio 4 — Ottimizzare i token *(libero)*

Il system prompt occupa token ad ogni chiamata.
Confrontate due versioni dello stesso system prompt:
- **Versione lunga**: almeno 100 parole
- **Versione corta**: massimo 20 parole, stesso significato

Quanti token risparmiate? E la qualità delle risposte cambia?

In [ ]:
# Esercizio 4 — confronto system prompt lungo vs corto

system_lungo = """
Sei l'assistente virtuale ufficiale di WiData Srl, una startup innovativa con sede a Sassari,
in Sardegna, specializzata in soluzioni IoT (Internet of Things) per il monitoraggio ambientale
e le smart cities. La nostra missione è rendere accessibile a comuni, aziende e cittadini il
monitoraggio in tempo reale della qualità dell'aria, del rumore e di altri parametri ambientali.
Quando rispondi devi sempre essere professionale, cortese e disponibile, usando un tono accessibile
ma competente. Collega ogni risposta, quando possibile, ai prodotti e ai servizi WiData. Non inventare
mai dati tecnici precisi o prezzi: in quel caso invita l'utente a contattare il reparto commerciale.
Rispondi sempre in italiano.
"""

system_corto = """
Sei l'assistente di WiData (startup IoT di Sassari, monitoraggio ambientale). Tono professionale, rispondi in italiano.
"""

domanda_test = "Quali prodotti offrite?"

def chiedi_con_system(system, domanda):
    return client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        system=system,
        messages=[{"role": "user", "content": domanda}],
    )

for nome, system in [("SYSTEM LUNGO", system_lungo), ("SYSTEM CORTO", system_corto)]:
    r = chiedi_con_system(system, domanda_test)
    costo = (r.usage.input_tokens / 1_000_000 * 1.0) + (r.usage.output_tokens / 1_000_000 * 5.0)
    print("=" * 55)
    print(nome)
    print("=" * 55)
    print(r.content[0].text)
    print(f"\n→ token input: {r.usage.input_tokens} | token output: {r.usage.output_tokens} | costo: ${costo:.6f}\n")

# Conclusione del gruppo:
# Il system prompt viene inviato (e pagato) ad OGNI chiamata: un prompt più corto
# risparmia token di input su ogni singola richiesta. Su pochi messaggi la differenza
# è trascurabile, ma su migliaia di chiamate diventa significativa.
# Vale la pena accorciarlo finché restano chiare le istruzioni essenziali (ruolo, tono,
# cosa NON fare). Se accorciandolo si perde un comportamento importante, meglio tenerlo.

---
## 📊 Preparate la presentazione

Avete **30 minuti** per completare gli esercizi e preparare **5 slide**.

Le slide devono rispondere a:
1. **Cos'è un token?** (con un esempio concreto che avete scoperto)
2. **Perché l'italiano costa di più?** (con i numeri che avete misurato)
3. **Quanto costa una conversazione reale?** (con i vostri risultati)
4. **Vale la pena ottimizzare il system prompt?** (con la vostra conclusione)
5. **La cosa più sorprendente** che avete scoperto

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*